# Telco Churn - Phase 2: Data Preprocessing and Feature Engineering

This notebook converts raw churn data into model-ready train/test datasets using a leakage-safe pipeline.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.feature_engineering import engineer_all_features
from src.preprocessing import (
    encode_binary_columns,
    encode_multiclass_columns,
    load_and_clean,
    scale_numeric_columns,
)

## Section 1: Load and Clean Raw Data

In [ ]:
data_path = Path("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = load_and_clean(str(data_path))

print("Shape after clean:", df.shape)
print("Columns:", len(df.columns))
print("TotalCharges dtype:", df["TotalCharges"].dtype)
print("Churn distribution:")
print(df["Churn"].value_counts(normalize=True).rename({0: "No", 1: "Yes"}))
df.head()

## Section 2: Train/Test Split (Leakage Guardrail)

In [ ]:
target_col = "Churn"
X = df.drop(columns=[target_col])
y = df[target_col]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print("X_train_raw:", X_train_raw.shape)
print("X_test_raw:", X_test_raw.shape)
print("y_train distribution:")
print(y_train.value_counts(normalize=True).rename({0: "No", 1: "Yes"}))
print("y_test distribution:")
print(y_test.value_counts(normalize=True).rename({0: "No", 1: "Yes"}))

## Section 3: Feature Engineering (Applied Post-Split)

In [ ]:
X_train_fe = engineer_all_features(X_train_raw)
X_test_fe = engineer_all_features(X_test_raw)

new_cols = [
    "tenure_group",
    "avg_monthly_charge",
    "service_count",
    "charge_per_service",
    "is_new_customer",
    "has_premium_support",
    "contract_risk_score",
]

print("Shape after feature engineering (train):", X_train_fe.shape)
print("Shape after feature engineering (test):", X_test_fe.shape)
print("New columns:", new_cols)
X_train_fe[new_cols].head()

## Section 4: Binary Encoding (Fit on Train, Apply to Test)

In [ ]:
binary_columns = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
]

X_train_bin, binary_mapping = encode_binary_columns(X_train_fe, binary_columns)
X_test_bin, _ = encode_binary_columns(X_test_fe, binary_columns, mapping=binary_mapping)

print("Binary mapping:")
print(binary_mapping)
X_train_bin[binary_columns].head()

## Section 5: Multi-class One-Hot Encoding (Fit on Train, Transform Test)

In [ ]:
multiclass_columns = [
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod",
    "tenure_group",
]

X_train_ohe, X_test_ohe, multiclass_encoder = encode_multiclass_columns(
    X_train_bin,
    X_test_bin,
    multiclass_columns,
)

print("X_train_ohe shape:", X_train_ohe.shape)
print("X_test_ohe shape:", X_test_ohe.shape)
print("OneHot encoded feature count:", len(multiclass_encoder.get_feature_names_out()))

## Section 6: Numeric Scaling (Fit on Train, Transform Test)

In [ ]:
numeric_columns_to_scale = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "avg_monthly_charge",
    "service_count",
    "charge_per_service",
    "contract_risk_score",
]

X_train_final, X_test_final, scaler = scale_numeric_columns(
    X_train_ohe,
    X_test_ohe,
    numeric_columns_to_scale,
)

print("X_train_final shape:", X_train_final.shape)
print("X_test_final shape:", X_test_final.shape)
X_train_final.head()

## Section 7: Verification Checks

In [ ]:
print("Train/Test feature columns aligned:", list(X_train_final.columns) == list(X_test_final.columns))
print("Any NaN in train:", X_train_final.isna().any().any())
print("Any NaN in test:", X_test_final.isna().any().any())
print("All train columns numeric:", all(np.issubdtype(dtype, np.number) for dtype in X_train_final.dtypes))
print("All test columns numeric:", all(np.issubdtype(dtype, np.number) for dtype in X_test_final.dtypes))
print("Finite train values:", np.isfinite(X_train_final.to_numpy(dtype=float)).all())
print("Finite test values:", np.isfinite(X_test_final.to_numpy(dtype=float)).all())

print("\nFinal class balance:")
print("y_train:")
print(y_train.value_counts(normalize=True).rename({0: "No", 1: "Yes"}))
print("y_test:")
print(y_test.value_counts(normalize=True).rename({0: "No", 1: "Yes"}))

## Section 8: Save Processed Datasets and Artifacts

In [ ]:
processed_dir = Path("../data/processed")
models_dir = Path("../models")
processed_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

X_train_path = processed_dir / "X_train.csv"
X_test_path = processed_dir / "X_test.csv"
y_train_path = processed_dir / "y_train.csv"
y_test_path = processed_dir / "y_test.csv"

X_train_final.to_csv(X_train_path, index=False)
X_test_final.to_csv(X_test_path, index=False)
y_train.to_frame(name="Churn").to_csv(y_train_path, index=False)
y_test.to_frame(name="Churn").to_csv(y_test_path, index=False)

joblib.dump(binary_mapping, models_dir / "binary_mapping.joblib")
joblib.dump(multiclass_encoder, models_dir / "multiclass_encoder.joblib")
joblib.dump(scaler, models_dir / "numeric_scaler.joblib")

print("Saved files:")
print("-", X_train_path)
print("-", X_test_path)
print("-", y_train_path)
print("-", y_test_path)
print("-", models_dir / "binary_mapping.joblib")
print("-", models_dir / "multiclass_encoder.joblib")
print("-", models_dir / "numeric_scaler.joblib")